In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# --- PATH ---
CSV_RESULT_PATH = '/Volumes/Extreme SSD/mcu_quake_big_stead_output/checkpoint_results_stead_full.csv'
STEAD_METADATA_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'

def fix_and_evaluate():
    print("[INFO] Memuat metadata asli untuk sinkronisasi label...")
    df_meta = pd.read_csv(STEAD_METADATA_PATH, usecols=['trace_name', 'trace_category'])
    df_results = pd.read_csv(CSV_RESULT_PATH)

    # Menghapus kolom y_true lama yang salah mapping
    df_results = df_results.drop(columns=['y_true'])

    # Gabungkan (Merge) berdasarkan trace_name untuk mendapatkan kategori asli yang benar
    df_final = pd.merge(df_results, df_meta, on='trace_name', how='left')

    # Mapping ulang yang lebih fleksibel (mencari kata 'earthquake')
    # Label 2 jika mengandung kata earthquake, selain itu 0 (noise)
    df_final['y_true'] = df_final['trace_category'].apply(
        lambda x: 2 if 'earthquake' in str(x).lower() else 0
    )

    y_true = df_final['y_true'].astype(int)
    y_pred = df_final['y_pred'].astype(int)

    print("\n" + "="*50)
    print("      HASIL EVALUASI PERBAIKAN (REAL METRICS)")
    print("="*50)
    print(f"Total Data: {len(df_final):,}")
    
    # Cek jumlah label asli
    print(f"Jumlah Asli Earthquake (2): {sum(y_true == 2):,}")
    print(f"Jumlah Asli Noise (0)     : {sum(y_true == 0):,}")
    print("-" * 50)

    print(classification_report(y_true, y_pred, target_names=['Noise (0)', 'Earthquake (2)']))
    print("="*50)

if __name__ == "__main__":
    fix_and_evaluate()

[INFO] Memuat metadata asli untuk sinkronisasi label...

      HASIL EVALUASI PERBAIKAN (REAL METRICS)
Total Data: 1,265,657
Jumlah Asli Earthquake (2): 1,030,231
Jumlah Asli Noise (0)     : 235,426
--------------------------------------------------
                precision    recall  f1-score   support

     Noise (0)       0.12      0.54      0.20    235426
Earthquake (2)       0.51      0.11      0.18   1030231

      accuracy                           0.19   1265657
     macro avg       0.31      0.32      0.19   1265657
  weighted avg       0.43      0.19      0.18   1265657

